# 05 — Tool comparison: ViraLift (tblastn) vs VAPiD (MAFFT nucleotide)

Same shared metric as `liftoff_compare.ipynb` / `lifton_compare.ipynb`: coordinate-only
(IoU >= 0.90 **or** both boundaries within +-6 bp), **codon check OFF for both tools**,
truth-anchored `R ∩ truth` denominator, and `lift_all_tblastn` is the real ViraLift engine.

**VAPiD is run in user-reference mode** so the comparison is apples-to-apples with the other
tools: the reference ViraLift uses is exported to a GenBank file and forced with `--f`, and its
NCBI accession is passed with `--r` (VAPiD needs a reference accession to skip its bundled BLAST
database; the record it fetches for that accession is discarded and replaced by our `--f` file).
This mirrors how Liftoff/LiftOn were given ViraLift's reference, and is disclosed in the methodology.

**VAPiD must be run where MAFFT is installed** (see `README_VAPID_RUN.md`). Everything else -- input
generation, ViraLift, and scoring -- runs anywhere. If MAFFT / vapid3.py are missing, cell 1 stops.

In [ ]:
from pathlib import Path
import sys, subprocess, shutil, tempfile, os, glob
import matplotlib.pyplot as plt, pandas as pd
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.SeqFeature import SeqFeature, FeatureLocation
from Bio.Seq import Seq

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from app.validation._shared.validation_utils import *
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.src.pipeline import PipelineConfig

OUTPUT_DIR = Path('outputs'); OUTPUT_DIR.mkdir(exist_ok=True)
INPUT_DIR  = OUTPUT_DIR / 'vapid_inputs'; INPUT_DIR.mkdir(exist_ok=True)
WORK_DIR   = OUTPUT_DIR / 'vapid_runs';   WORK_DIR.mkdir(exist_ok=True)
CFG = PipelineConfig()
TOOL_COLORS = {'ViraLift': '#2f7d4f', 'VAPiD': '#7d2f6f'}

# --- locate VAPiD (env VAPID_DIR only if it actually contains vapid3.py, else sibling of repo) ---
_env = os.environ.get('VAPID_DIR', '')
VAPID_DIR = Path(_env) if _env and (Path(_env) / 'vapid3.py').exists() else (ROOT.parent / 'VAPiD')
VAPID_PY  = VAPID_DIR / 'vapid3.py'
SBT       = VAPID_DIR / 'example.sbt'

# --- locate MAFFT; if not on the kernel PATH (e.g. installed in conda base while a venv is
#     active), search common conda/homebrew locations and PREPEND its dir to PATH so both this
#     cell AND VAPiD's `mafft` subprocess can find it (VAPiD calls mafft via shell) ---
def _find_mafft():
    m = shutil.which('mafft')
    if m:
        return m
    bases = [os.environ.get('CONDA_PREFIX', ''),
             '/opt/homebrew/Caskroom/miniforge/base', '/opt/homebrew', '/usr/local',
             os.path.expanduser('~/miniforge3'), os.path.expanduser('~/miniconda3'),
             os.path.expanduser('~/anaconda3'), os.path.expanduser('~/mambaforge')]
    cands = []
    for b in bases:
        if not b:
            continue
        cands += glob.glob(os.path.join(b, 'bin', 'mafft'))
        cands += glob.glob(os.path.join(b, 'envs', '*', 'bin', 'mafft'))
    for c in cands:
        if os.path.exists(c):
            os.environ['PATH'] = os.path.dirname(c) + os.pathsep + os.environ.get('PATH', '')
            return c
    return None
MAFFT = _find_mafft()

print('vapid3.py :', VAPID_PY if VAPID_PY.exists() else f'NOT FOUND -> git clone VAPiD into {ROOT.parent}')
print('mafft     :', MAFFT or 'NOT FOUND -> conda install -c bioconda mafft   (or brew install mafft)')
print('sbt       :', SBT if SBT.exists() else 'NOT FOUND (VAPiD ships example.sbt)')
assert VAPID_PY.exists() and SBT.exists(), 'Point VAPID_DIR at your VAPiD checkout.'
assert MAFFT, ('MAFFT not found. Installed in conda base but running a venv? Symlink it in: '
               'ln -sf $(conda run -n base which mafft) "$VIRTUAL_ENV/bin/mafft"  (then restart kernel).')

In [ ]:
# ---------------- helpers ----------------
def build_reference_gbk(bundle, path):
    """Export ViraLift's reference (sequence + features) as GenBank for VAPiD --f, so VAPiD lifts
    from the SAME reference and copies the SAME canonical gene names."""
    rec, ft = bundle['record'], bundle['feature_type']
    sr = SeqRecord(Seq(str(rec.seq)), id=rec.id, name=rec.id[:16],
                   description=f"{bundle.get('virus_name') or rec.id} reference for VAPiD (ViraLift bundle)")
    sr.annotations['molecule_type'] = 'RNA'
    for f in bundle['features']:
        loc = FeatureLocation(int(f['start']) - 1, int(f['end']), strand=1 if f['strand'] == '+' else -1)
        sr.features.append(SeqFeature(loc, type=ft, qualifiers={'gene': [f['name']], 'product': [f['name']]}))
    SeqIO.write(sr, str(path), 'genbank')

def write_targets_and_meta(records, fasta_path, meta_path):
    with open(fasta_path, 'w') as fh:
        for r in records: fh.write(f'>{r.id}\n{str(r.seq)}\n')
    with open(meta_path, 'w') as mh:      # non-interactive metadata (values irrelevant to coordinates)
        mh.write('strain,collection-date,country,coverage\n')
        for r in records: mh.write(f'{r.id},2020,USA,50\n')

def _clean(tok):
    return tok.replace('<', '').replace('>', '').strip()

def vapid_tbl_to_rows(record_id, tbl_path):
    """Parse a VAPiD .tbl (NCBI feature table) into prediction rows, aggregating multi-interval
    features by name (min start / max end); same schema as liftoff_gff_to_rows."""
    p = Path(tbl_path)
    if not p.exists(): return []
    feats, cur = [], None
    for line in p.read_text().splitlines():
        if line.startswith('>Feature') or not line.strip(): continue
        c = line.split('\t')
        if len(c) >= 3 and c[0].strip() and c[2].strip():
            try: s, e = int(_clean(c[0])), int(_clean(c[1]))
            except ValueError:
                cur = None; continue
            cur = [min(s, e), max(s, e), '+' if s <= e else '-']
        elif len(c) >= 2 and c[0].strip() and c[1].strip() and (len(c) < 3 or not c[2].strip()):
            if cur:
                try: s2, e2 = int(_clean(c[0])), int(_clean(c[1]))
                except ValueError: continue
                cur[0] = min(cur[0], s2, e2); cur[1] = max(cur[1], s2, e2)
        elif len(c) >= 5 and c[3].strip() in ('product', 'gene') and cur is not None:
            feats.append((c[4].strip(), cur[0], cur[1], cur[2])); cur = None
    agg = {}
    for name, s, e, strand in feats:
        if name not in agg: agg[name] = [s, e, strand]
        else: agg[name][0] = min(agg[name][0], s); agg[name][1] = max(agg[name][1], e)
    return [{'record_id': record_id, 'method': 'vapid', 'pred_name': n,
             'pred_start': v[0], 'pred_end': v[1], 'strand': v[2],
             'has_start_codon': None, 'has_stop_codon': None, 'in_frame': None}
            for n, v in agg.items()]

def run_vapid_one(record, ref_gbk, ref_acc, meta_csv, workdir, all_flag=False, timeout=600, retries=3):
    """Run real VAPiD on a SINGLE record (own strain folder under workdir) so we get per-record
    progress. Requires MAFFT + internet (VAPiD does a discarded Entrez lookup for --r). Retries up
    to `retries` times because that per-record NCBI fetch can flake transiently and leave no .tbl.
    Returns the .tbl path (may still not exist if VAPiD genuinely failed)."""
    import shutil as _sh
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    tbl = workdir / record.id / f'{record.id}.tbl'
    fa = workdir / f'{record.id}.fasta'
    cmd = [sys.executable, str(VAPID_PY), str(fa.resolve()), str(SBT.resolve()),
           '--r', ref_acc, '--f', str(Path(ref_gbk).resolve()),
           '--metadata_loc', str(Path(meta_csv).resolve())]
    if all_flag: cmd.append('--all')  # transfer non-CDS (mat_peptide) features, e.g. FMD
    for _ in range(max(1, retries)):
        _sh.rmtree(workdir / record.id, ignore_errors=True)      # clean any partial prior attempt
        fa.write_text(f'>{record.id}\n{str(record.seq)}\n')
        try:
            subprocess.run(cmd, cwd=str(workdir), stdout=subprocess.DEVNULL,
                           stderr=subprocess.DEVNULL, timeout=timeout, check=False)
        except subprocess.TimeoutExpired:
            pass
        if tbl.exists():
            break
    return tbl

def coverage_rows(virus, tool, acc, truth, preds):
    """Truth-anchored, coordinate-only (codon check OFF for both tools) -- identical to
    liftoff_compare.ipynb / gatu_50case so all units report the same buckets on the same lift."""
    if preds:
        cmp = compare_predictions_to_truth(preds, truth, codon_required_names=set())
        correct = set(cmp.loc[cmp['coord_correct'], 'pred_name']) if 'coord_correct' in cmp.columns else set()
        exact   = set(cmp.loc[cmp['exact_match'], 'pred_name']) if 'exact_match' in cmp.columns else set()
    else:
        correct, exact = set(), set()
    return [{'virus': virus, 'tool': tool, 'accession': acc, 'gene': t['name'],
             'found': t['name'] in correct, 'exact': t['name'] in exact} for t in truth]

In [ ]:
# Live X/100 progress per virus. Uses tqdm if installed (pretty bar: pip install tqdm),
# otherwise a built-in fallback that prints an updating "desc: n/total" line in real time.
try:
    from tqdm.auto import tqdm
except Exception:
    class tqdm:
        def __init__(self, it, total=None, desc='', unit=''):
            self.it = list(it); self.total = total if total is not None else len(self.it)
            self.desc = desc; self.n = 0; self._post = ''
        def __iter__(self):
            print(f'\r{self.desc}: 0/{self.total}', end='', flush=True)
            for x in self.it:
                yield x
                self.n += 1
                print(f'\r{self.desc}: {self.n}/{self.total}  {self._post}', end='', flush=True)
            print()
        def set_postfix(self, **k):
            self._post = '  '.join(f'{a}={b}' for a, b in k.items())

DATASETS = [
    ('PRRS', DATA / 'PRRS' / 'PRRS_ref_test.gb', DATA / 'PRRS' / 'PRRS_100seq_anno.gb', 'PQ623173.1'),
    ('FMD',  DATA / 'FMD'  / 'FMD_ref_test.gb',  DATA / 'FMD'  / 'FMD_100seq_anno.gb',  'FJ175661.1'),
    ('PED',  DATA / 'PED'  / 'PED_ref_1.gb',     DATA / 'PED'  / 'PED_100seqs.gb',      'PZ105934.1'),
]
SKIP_VAPID_IF_DONE = True   # reuse existing .tbl files; set False to force VAPiD to re-run

cov, run_rows, detail = [], [], []
for virus, ref_path, query_path, ref_acc in DATASETS:
    bundle = load_reference_bundle(ref_path)
    ref_names = sorted({f['name'] for f in bundle['features']})       # evaluable set R
    validate_codons = bundle['feature_type'] == 'CDS'
    records = load_genbank_records(query_path)

    ref_gbk = INPUT_DIR / f'{virus}_ref.gbk'; build_reference_gbk(bundle, ref_gbk)
    _targets = INPUT_DIR / f'{virus}_targets.fasta'; meta = INPUT_DIR / f'{virus}_meta.csv'
    write_targets_and_meta(records, _targets, meta)
    workdir = WORK_DIR / virus

    n_vapid_ok = 0
    bar = tqdm(records, desc=f'{virus} VAPiD', unit='rec')
    for record in bar:
        # --- run real VAPiD for THIS record (or reuse its .tbl) ---
        tbl = workdir / record.id / f'{record.id}.tbl'
        if not (SKIP_VAPID_IF_DONE and tbl.exists()):
            tbl = run_vapid_one(record, ref_gbk, ref_acc, meta, workdir, all_flag=(bundle['feature_type'] != 'CDS'))

        truth, _ = parse_truth_features(record, bundle['alias_lookup'], bundle['feature_type'],
                                        filter_nested=False, target_names=ref_names, keep_extra_names=[])
        truth = dedupe_truth_by_name(truth)
        if not truth:
            continue
        v_lift = lift_all_tblastn(
            ref_features=bundle['features'], ref_record=bundle['record'], query_record=record,
            min_coverage=CFG.min_coverage, min_identity=CFG.min_identity,
            evalue=CFG.evalue, rescue_window=CFG.rescue_window, validate_codons=validate_codons)
        v_preds = lifted_to_rows(record.id, v_lift, 'viralift')
        p_preds = vapid_tbl_to_rows(record.id, tbl)
        if p_preds:
            n_vapid_ok += 1
        cov += coverage_rows(virus, 'ViraLift', record.id, truth, v_preds)
        cov += coverage_rows(virus, 'VAPiD',    record.id, truth, p_preds)
        for tool, preds in (('ViraLift', v_preds), ('VAPiD', p_preds)):
            if not preds: continue
            d = compare_predictions_to_truth(preds, truth, codon_required_names=set())
            d = d[d['name_match']].copy(); d.insert(0, 'virus', virus); d.insert(1, 'tool', tool)
            detail.append(d)
        try:
            bar.set_postfix(vapid_ok=n_vapid_ok)      # live count of records VAPiD annotated
        except Exception:
            pass

    run_rows.append({'virus': virus, 'query_records': len(records),
                     'ref_feature_type': bundle['feature_type'],
                     'records_vapid_produced_output': n_vapid_ok})

cov_df = pd.DataFrame(cov); cov_df.to_csv(OUTPUT_DIR / 'vapid_coverage_per_gene.tsv', sep='\t', index=False)
detail_df = pd.concat(detail, ignore_index=True) if detail else pd.DataFrame()
detail_df.to_csv(OUTPUT_DIR / 'vapid_per_feature.tsv', sep='\t', index=False)
pd.DataFrame(run_rows)

In [ ]:
# ---------------- summary (coord_pct per virus per tool) ----------------
summary = (cov_df.groupby(['virus', 'tool'])
           .agg(truth_genes=('found', 'size'), correct=('found', 'sum'), exact=('exact', 'sum'))
           .reset_index())
summary['coord_pct'] = (summary['correct'] / summary['truth_genes'] * 100).round(2)
summary['exact_pct'] = (summary['exact']   / summary['truth_genes'] * 100).round(2)
summary.to_csv(OUTPUT_DIR / 'vapid_summary.tsv', sep='\t', index=False)
wide = summary.pivot(index='virus', columns='tool', values='coord_pct')
wide.to_csv(OUTPUT_DIR / 'vapid_summary_wide.tsv', sep='\t')
print(wide)
summary

In [ ]:
# ---------------- coordinate accuracy figure + failure detail ----------------
fig, ax = plt.subplots(figsize=(6.2, 3.6))
piv = summary.pivot(index='virus', columns='tool', values='coord_pct')[['ViraLift', 'VAPiD']]
piv.plot(kind='bar', ax=ax, color=[TOOL_COLORS['ViraLift'], TOOL_COLORS['VAPiD']], width=0.75)
ax.set_ylabel('Coordinate-correct (% of R cap truth)'); ax.set_ylim(0, 105)
ax.set_title('ViraLift vs VAPiD -- coordinate accuracy'); ax.legend(title='', loc='lower right')
for c in ax.containers: ax.bar_label(c, fmt='%.1f', fontsize=7, padding=2)
plt.xticks(rotation=0); plt.tight_layout(); plt.savefig(OUTPUT_DIR / 'vapid_accuracy.png', dpi=150); plt.show()

if not detail_df.empty:
    keep = [c for c in ['virus','tool','pred_name','delta_start','delta_end','iou','coord_correct','exact_match'] if c in detail_df.columns]
    detail_df[keep].to_csv(OUTPUT_DIR / 'vapid_failure_detail.tsv', sep='\t', index=False)
unmapped = (cov_df[cov_df['tool'] == 'VAPiD'].groupby('virus')
            .agg(truth_genes=('found','size'), correct=('found','sum')).reset_index())
unmapped['not_emitted_pct'] = ((unmapped['truth_genes']-unmapped['correct'])/unmapped['truth_genes']*100).round(2)
unmapped.to_csv(OUTPUT_DIR / 'vapid_not_emitted.tsv', sep='\t', index=False)
unmapped

## Operating range (accuracy vs divergence)

The headline is **robustness at divergence**, not average accuracy — same analysis as
`lifton_compare.ipynb`. Records are binned by `mean_protein_identity` (mean identity of ViraLift's
own tblastn-lifted genes vs the reference, a free per-record divergence proxy). The `<70` bin is the
cross-genotype set (e.g. PRRSV-1 vs a PRRSV-2 reference) where nucleotide lifters are expected to
collapse. Reuses the existing VAPiD `.tbl` files — VAPiD is **not** re-run.

> **Fairness caveat — do NOT report VAPiD's `<70` number as its cross-genotype capability.** This cell scores VAPiD in *forced-reference* mode (our fixed reference via `--f`). VAPiD's *default* auto-selects a best-BLAST-hit reference per genome, so for a cross-genotype query it would pick a matching-genotype reference and lift fine. Forcing our reference here pushes VAPiD into a lift its design avoids, so a low `<70` value reflects the forced setup, not VAPiD. The operating-range figure (Fig 2) therefore compares ViraLift vs LiftOn/Liftoff (fixed-reference tools by design); VAPiD is reported only on the same-genotype benchmark (Table 2). VAPiD's real divergence strategy — switching references per genome — is discussed in the text, because it reintroduces the per-record name/coordinate inconsistency that ViraLift is built to remove.

In [ ]:
DATASETS = [
    ('PRRS', DATA/'PRRS'/'PRRS_ref_test.gb', DATA/'PRRS'/'PRRS_100seq_anno.gb'),
    ('FMD',  DATA/'FMD'/'FMD_ref_test.gb',   DATA/'FMD'/'FMD_100seq_anno.gb'),
    ('PED',  DATA/'PED'/'PED_ref_1.gb',      DATA/'PED'/'PED_100seqs.gb'),
]
covd, divergence = [], []
for virus, ref_path, query_path in DATASETS:
    bundle = load_reference_bundle(ref_path)
    ref_names = sorted({f['name'] for f in bundle['features']})
    validate_codons = bundle['feature_type'] == 'CDS'
    wd = WORK_DIR / virus
    for record in load_genbank_records(query_path):
        truth, _ = parse_truth_features(record, bundle['alias_lookup'], bundle['feature_type'],
                                        filter_nested=False, target_names=ref_names, keep_extra_names=[])
        truth = dedupe_truth_by_name(truth)
        if not truth: continue
        v_lift = lift_all_tblastn(ref_features=bundle['features'], ref_record=bundle['record'], query_record=record,
                                  min_coverage=CFG.min_coverage, min_identity=CFG.min_identity,
                                  evalue=CFG.evalue, rescue_window=CFG.rescue_window, validate_codons=validate_codons)
        v_preds = lifted_to_rows(record.id, v_lift, 'viralift')
        p_preds = vapid_tbl_to_rows(record.id, wd / record.id / f'{record.id}.tbl')
        covd += coverage_rows(virus, 'ViraLift', record.id, truth, v_preds)
        covd += coverage_rows(virus, 'VAPiD',    record.id, truth, p_preds)
        ids = [f.identity for f in v_lift if getattr(f, 'identity', None)]
        divergence.append({'accession': record.id,
                           'mean_protein_identity': round(sum(ids) / len(ids), 2) if ids else None})

covd_df = pd.DataFrame(covd); div_df = pd.DataFrame(divergence)
div_df.to_csv(OUTPUT_DIR / 'vapid_divergence.tsv', sep='\t', index=False)

bins, labels = [0, 70, 80, 90, 95, 101], ['<70', '70-80', '80-90', '90-95', '>=95']
per_rec = (covd_df.groupby(['accession', 'tool']).agg(truth=('found', 'size'), correct=('found', 'sum')).reset_index()
           .merge(div_df, on='accession', how='left'))
per_rec['identity_bin'] = pd.cut(per_rec['mean_protein_identity'], bins=bins, labels=labels, right=False)
by_div = (per_rec.groupby(['identity_bin', 'tool'], observed=True)
          .agg(truth_genes=('truth', 'sum'), correct=('correct', 'sum')).reset_index())
by_div['accuracy_pct'] = (by_div['correct'] / by_div['truth_genes'] * 100).round(2)
by_div.to_csv(OUTPUT_DIR / 'vapid_accuracy_by_divergence.tsv', sep='\t', index=False)

print('records per identity bin:')
print(div_df.assign(identity_bin=pd.cut(div_df['mean_protein_identity'], bins=bins, labels=labels, right=False))
      .groupby('identity_bin', observed=True).size().to_string())
print('\naccuracy by divergence (VAPiD vs ViraLift):')
print(by_div.pivot(index='identity_bin', columns='tool', values='accuracy_pct').to_string())